In [ ]:
# imports

import math

import numpy as np
import pandas as pd
from bokeh.io import output_notebook

from omni_drive import (
    Angle,
    Clock,
    Distance,
    DriveVector,
    Kalman2d,
    OmniDrive,
    PoseRecorder,
    SensorRecorder,
    Time,
)
from omni_drive.live import LivePlot

output_notebook(hide_banner=True)


In [ ]:
robot = OmniDrive()

In [ ]:
robot.print_status()

## Część 1: Ruch i odometria

### Model ruchu robota omnikierunkowego

<small>Robot ma trzy <b>koła omnikierunkowe</b> (omni wheels) rozmieszczone co 120°. Na obwodzie każdego koła znajdują się bierne rolki: koło napędza robota tylko w kierunku stycznym do swojego obwodu, a w kierunku prostopadłym swobodnie przetacza się na rolkach. Dzięki temu robot może jechać w dowolnym kierunku bez skręcania i niezależnie obracać się wokół własnej osi. Taki napęd nazywamy <b>holonomicznym</b> (holonomic).</small>

<div style="text-align: center; margin: 20px 0;">
  <img src="lab/img/omni_wheel_rolling.gif" alt="Koło omnikierunkowe" style="max-width: 380px; width: 100%; height: auto;" />
</div>

<small>Ruch robota zadajemy wektorem `DriveVector(velocity, heading, omega)`, który łączy <b>dwa niezależne rodzaje ruchu</b>:</small>

1. <small><b>Ruch postępowy</b> (translacja) – `velocity` to prędkość jako ułamek prędkości maksymalnej (0–1), a `heading` to kierunek jazdy (0° – przód, 90° – lewo, 180° – tył). Robot przesuwa się, nie zmieniając orientacji: aby pojechać w bok lub do tyłu, nie musi się obracać.</small>
2. <small><b>Ruch obrotowy</b> (rotacja) – `omega` to prędkość obrotu jako ułamek prędkości maksymalnej; wartość dodatnia oznacza obrót przeciwnie do ruchu wskazówek zegara. Przy `omega = 1.0` robot obraca się z prędkością </small>$\omega_{\text{max}} \approx 3{,}9$<small> rad/s.</small>

<small>Oba ruchy można wykonywać jednocześnie, ponieważ prędkość obwodu każdego koła jest po prostu sumą ich udziałów:</small>

$$w_i = \underbrace{-\sin(\beta_i)\, v_x + \cos(\beta_i)\, v_y}_{\text{ruch postępowy}} + \underbrace{L\, \omega}_{\text{ruch obrotowy}}, \qquad \beta_i \in \{60°, 180°, 300°\},$$

<small>gdzie </small>$(v_x, v_y)$<small> to prędkość w układzie robota (oś </small>$x$<small> do przodu, oś </small>$y$<small> w lewo), a </small>$L$<small> to odległość kół od środka robota. Przy ruchu obrotowym wszystkie koła kręcą się z tą samą prędkością, a przy postępowym – z różnymi, zależnie od kierunku jazdy. W Zadaniu 1 zaimplementujesz te obliczenia samodzielnie.</small>

#### Układ odniesienia

<small>Gdy robot obraca się w trakcie jazdy, trzeba zdecydować, względem czego liczony jest kierunek `heading`. Decyduje o tym flaga `robot.field_centric`:</small>

* <small>`True` (<b>field-centric</b>, domyślnie) – kierunek liczony jest w układzie pomieszczenia. Polecenie trzeba na bieżąco obracać o aktualny kąt robota, dzięki czemu obracający się robot jedzie po linii prostej.</small>
* <small>`False` (<b>robot-centric</b>) – kierunek liczony jest względem przodu robota, więc obraca się razem z nim, a tor jazdy zakrzywia się.</small>

<div style="text-align: center; margin: 20px 0;">
  <img src="assets/field_vs_robot_centric.svg" alt="Porównanie trybów field-centric i robot-centric" style="max-width: 600px; width: 100%; height: auto;" />
</div>

**Zadanie 1 – Kinematyka robota**

<small>W tym zadaniu nie korzystamy z `DriveVector` ani `drive_for` – sami przeliczamy ruch robota na prędkości kół, używając wyłącznie funkcji trygonometrycznych (`math.sin`, `math.cos`) i ustawiając koła bezpośrednio (`robot.wheel_N.speed`). Nie korzystaj też z modułu `omni_drive.kinematics`.</small>

<small><b>a)</b> Uzupełnij funkcję `wheel_speeds(vx, vy, omega)`, która na podstawie wzoru z poprzedniej sekcji zwraca prędkości trzech kół </small>$(w_1, w_2, w_3)$<small>. Wszystkie wielkości są ułamkami prędkości maksymalnej, więc składnik obrotowy </small>$L\,\omega$<small> to po prostu `omega`. Kąty </small>$\beta_i$<small> zamień na radiany.</small>

In [ ]:
BETA = [math.radians(60), math.radians(180), math.radians(300)]  # położenie kół w1, w2, w3

def wheel_speeds(vx, vy, omega):
    # prędkość każdego koła: udział ruchu postępowego + udział ruchu obrotowego
    ...
    return ...

# test: jazda do przodu, jazda w lewo, obrót w miejscu
for (vx, vy, omega), expected in [
    ((1.0, 0.0, 0.0), (-0.866, 0.0, 0.866)),
    ((0.0, 1.0, 0.0), (0.5, -1.0, 0.5)),
    ((0.0, 0.0, 1.0), (1.0, 1.0, 1.0)),
]:
    got = wheel_speeds(vx, vy, omega)
    assert np.allclose(got, expected, atol=1e-3), f"wheel_speeds({vx}, {vy}, {omega}) = {got}, oczekiwano {expected}"
print("wheel_speeds: OK")

<small><b>b)</b> Zaprogramuj przejazd, który wykorzystuje oba rodzaje ruchu. Gotowa funkcja `drive_raw` rozkłada polecenie (`velocity`, `heading`) na składowe </small>$v_x = v\cos\theta$<small>, </small>$v_y = v\sin\theta$<small> (gdzie </small>$\theta = \text{heading} - \text{yaw}$<small>, czyli tryb field-centric) i ustawia koła za pomocą Twojej funkcji `wheel_speeds`. Uzupełnij funkcję `drive_there_and_back`, tak aby robot:</small>

1. <small>pojechał <b>do przodu</b> ruchem postępowym (`omega = 0`) przez czas `DURATION`,</small>
2. <small>zatrzymał się na chwilę,</small>
3. <small>wrócił <b>do tyłu</b> tą samą drogą, w trakcie powrotu wykonując <b>jeden pełny obrót</b> wokół własnej osi.</small>

<small>Prędkość obrotu dobierz tak, aby w czasie </small>$t$<small> robot obrócił się dokładnie o </small>$2\pi$<small>:</small>

$$\omega = \frac{2\pi}{t \cdot \omega_{\text{max}}}$$

<div class="alert alert-block alert-success">
  <small><b>Wskazówka:</b> Prędkość koła to suma udziałów obu ruchów, a prędkości poza zakresem [−1, 1] są obcinane, więc <code>velocity + omega</code> nie powinno przekraczać 1 – inaczej robot nie przejedzie zadanej drogi. Wybierz niewielką prędkość (np. 0.2) i odpowiednio długi czas. Klatka ma wymiary 78 × 78 cm: w czasie <i>t</i> robot przejeżdża ok. <code>velocity</code> · 0,39 m/s · <i>t</i>, więc ogranicz przejazd do ok. 20 cm, a robota ustaw ok. 10 cm za środkiem klatki.</small>
</div>

In [ ]:
OMEGA_MAX = 3.89  # prędkość obrotu [rad/s] przy omega = 1.0

VELOCITY = ...
DURATION = Time.s(...)

def set_wheels(w1, w2, w3):
    # batch() wysyła prędkości wszystkich trzech kół jednocześnie
    with robot.batch():
        robot.wheel_1.speed = w1
        robot.wheel_2.speed = w2
        robot.wheel_3.speed = w3

def drive_raw(velocity, heading, omega, duration):
    for t in Clock(duration=duration):
        theta = heading.radians - robot.yaw  # kierunek jazdy w układzie robota [rad]
        vx = velocity * math.cos(theta)
        vy = velocity * math.sin(theta)
        set_wheels(*wheel_speeds(vx, vy, omega))
    robot.stop()

def drive_there_and_back():
    # 1. Ruch postępowy: jazda do przodu bez obrotu
    drive_raw(...)
    robot.pause_for(...)

    # 2. Ruch złożony: powrót do tyłu z jednoczesnym pełnym obrotem
    omega = ...
    drive_raw(...)
    robot.pause_for(...)

robot.reset_pose()
robot.reset_yaw()

rec = PoseRecorder(
    odometry=lambda: (robot.pose.x.meters, robot.pose.y.meters),
    title="Model ruchu",
)

with LivePlot(rec), robot:
    drive_there_and_back()

print(f"Odometria po przejeździe: {robot.pose}")

<div class="alert alert-block alert-info">
  <small><b>Pytania:</b> Czy tor powrotu był prosty, mimo że robot się obracał? Czy robot wrócił do punktu startu i czy jest zwrócony przodem w tę samą stronę co na początku? Porównaj kąt <code>yaw</code> z odometrii z rzeczywistym ustawieniem robota – który rodzaj ruchu, postępowy czy obrotowy, bardziej pogarsza dokładność odometrii?</small>
</div>

**Zadanie 2 – Test dryfu odometrii**

<small>Sprawdźmy, jak odometria zachowuje się w praktyce. Wykonaj test dryfu kół, realizując przejazd po zamkniętej trasie kwadratu – wyłącznie ruchem postępowym, bez obrotu (`omega = 0`). Klatka ma wymiary 78 × 78 cm, dlatego bok kwadratu ogranicz do ok. 20 cm (przy `velocity = 0.5` to ok. 1 s jazdy), a robota ustaw ok. 10 cm za środkiem klatki i 10 cm na prawo od niego. Uzupełnij funkcję `drive_square`:</small>


In [ ]:
def drive_square():
    robot.drive_for(...)
    robot.pause_for(...)
    ...

robot.reset_pose()

for i in range(1):  # zwiększ liczbę przejść gdy uzupełnisz definicję drive_square
    drive_square()
    print(f"Przejście nr {i + 1}: Odometria: {robot.pose}")


<div class="alert alert-block alert-info">
  <small><b>Pytanie:</b> Czy robot wrócił w miejsce początkowe? Przed przejazdem zaznacz pozycję startową (np. taśmą przy kołach), a po przejazdach zmierz <b>miarą zwijaną</b>, o ile robot rzeczywiście się przesunął. Porównaj ten wynik z położeniem raportowanym przez odometrię (<code>robot.pose</code>) – czy odometria „wie”, gdzie robot naprawdę się znalazł?</small>
</div>

### Położenie na podstawie LiDARu

<small>Jak już wiesz, oprócz trzech par czujników odległości zamontowanych co 90°, robot posiada również skaner laserowy LiDAR, który mierzy odległość do 680 punktów w polu widzenia 240°.</small>

<small>Będziemy korzystać z pomiarów LiDAR-u dokładnie w tych samych kierunkach, w które skierowane są pozostałe sensory. Biblioteka udostępnia w tym celu gotowe właściwości API, które wyznaczają średnią arytmetyczną z 5 punktów lasera w osi każdego z czujników:</small>

* <small>`robot.lidar.beam_left` – wiązka w lewo (+90°),</small>
* <small>`robot.lidar.beam_front` – wiązka na wprost (0°),</small>
* <small>`robot.lidar.beam_right` – wiązka w prawo (-90°).</small>

<small>Uśrednienie kilku sąsiednich punktów skutecznie redukuje szum pojedynczego pomiaru lasera.</small>

<small>Znając przesunięcie robota względem trzech punktów pomiarowych (ścian: lewej, przedniej i prawej) potrafimy z dużą dokładnością określić jego pozycję w osi Y oraz trochę mniej dokładnie w osi X:</small>

<div style="display: flex; gap: 20px; justify-content: center; align-items: flex-start; margin: 20px 0;">
  <div style="flex: 1; text-align: center; max-width: 380px;">
    <img src="lab/img/lidar_walls_start.svg" alt="Pozycja startowa" style="width: 100%; height: auto;" />
  </div>
  <div style="flex: 1; text-align: center; max-width: 380px;">
    <img src="lab/img/lidar_walls_moved.svg" alt="Po przesunięciu" style="width: 100%; height: auto;" />
  </div>
</div>

_Gdy robot przesunie się o $x$ do przodu, odległość do przedniej ściany zmaleje o $x$, a gdy przesunie się o $y$ w lewo, odległość do lewej ściany zmaleje o $y$, a do prawej wzrośnie o $y$. Położenie względem startu to zatem:_

$$
x = f_0 - f, \qquad y = \frac{(l_0 - l) - (r_0 - r)}{2},
$$

<small>gdzie:</small>

* $f_0$, $l_0$, $r_0$ <small>to odległości do przedniej, lewej i prawej ściany zmierzone na starcie (przed ruszeniem robota),</small>
* $f$, $l$, $r$ <small>to aktualne odległości odczytywane w trakcie jazdy z wiązek LiDAR-u (odpowiednio: `beam_front`, `beam_left`, `beam_right`).</small>


<div class="alert alert-block alert-warning">
  <small>
  <b>Uwaga:</b> Takie położenie nie dryfuje, ponieważ ściany się nie przesuwają, ale uproszczenie to działa tylko w znanym, prostokątnym otoczeniu i przy stałej orientacji robota. W praktyce roboty zwykle poruszają się w dowolnym otoczeniu – budują wtedy z kolejnych skanów mapę i dopasowują do niej każdy nowy skan (scan matching); zasada pozostaje ta sama, ale ściany i punkty odniesienia są wyznaczane bezpośrednio z danych.
  </small>
</div>


**Zadanie 3 – Odometria czy LiDAR**

<small>Wykorzystamy funkcję `drive_square` z poprzedniego zadania. Ustaw robota w klatce (78 × 78 cm) w tym samym miejscu co w Zadaniu 2, tak aby podczas przejazdu nie zbliżał się do ścian.</small>

<small>Wykonamy trzy przejazdy, analizując pozycję robota raportowaną przez odometrię oraz obliczaną na podstawie danych z LiDAR-u.</small>

<small>Zaimplementuj logikę pozwalającą wyznaczać pozycję x oraz y za pomocą LiDAR-u.</small>


In [ ]:
f0, l0, r0 = None, None, None

def lidar_reset():
    global f0, l0, r0
    f0 = robot.lidar.beam_front
    l0 = robot.lidar.beam_left
    r0 = robot.lidar.beam_right

def lidar_get_x():
    f = robot.lidar.beam_front
    dx = ...
    return Distance.m(dx)

def lidar_get_y():
    ...
    dy = ...
    return Distance.m(dy)


In [ ]:
robot.reset_pose()
lidar_reset()

rec = PoseRecorder(
    odometry=lambda: (robot.pose.x.meters, robot.pose.y.meters),
    lidar=lambda: (lidar_get_x().meters, lidar_get_y().meters),
    title="Odometria vs LiDAR"
)

def print_pos(label: str):
    pose = robot.pose
    px, py = pose.x.meters, pose.y.meters
    lx, ly = lidar_get_x().meters, lidar_get_y().meters

    d = math.hypot(px - lx, py - ly)
    print(f"{label}: Odometria: ({px:.3f}, {py:.3f}), LiDAR: ({lx:.3f}, {ly:.3f}), Różnica: {d:.3f} m")
    return d

print_pos("Pozycja startowa")

final_drift = 0.0
with LivePlot(rec), robot:
    for i in range(3):
        drive_square()

        # Sprawdzamy wartości oraz różnicę po każdym przejściu pętli
        final_drift = print_pos(f"Przejście nr {i + 1}")


<div class="alert alert-block alert-info">
  <small><b>Pytania:</b> Co jest dokładniejsze: odometria czy pozycja obliczona na podstawie LiDAR-u? Jak duża jest różnica? Dlaczego na zamkniętej trasie część błędu odometrii się znosi?</small>
</div>

<div class="alert alert-block alert-success">
  <small><b>Wskazówka:</b> Pozycja z LiDAR-u to nasz niezależny punkt odniesienia. Nie używamy jej do obliczeń &ndash; służy wyłącznie na wykresach do oceny błędu odometrii i skuteczności kolejnych algorytmów.</small>
</div>

**Czym jest szum procesu $Q$?**

<small>Zaobserwowana w teście różnica między odometrią a LiDAR-em na mecie (`final_drift`) to wynik kumulacji drobnych poślizgów kół na każdym kroku pętli sterowania. Pojedynczy błąd na jeden krok pętli (odometrii) jest rzędu ułamka milimetra, ale w miarę upływu czasu błędy sumują się, powodując narastający dryf pozycji robota.</small>

<small>W modelu ruchu robota tę losową niepewność opisujemy jako <b>wariancję szumu procesu</b> </small>$Q$<small> przypadającą na pojedynczy krok odometrii. Skoro wariancje niezależnych błędów sumują się z każdym krokiem, po </small>$N$<small> próbkach zarejestrowanych przez `rec` całkowita wariancja dryfu wynosi:</small>

$$\sigma^2_{\text{całk}} \approx N \cdot Q \implies Q \approx \frac{\Delta_{\text{dryf}}^2}{N}$$

<small>Wykorzystaj zarejestrowaną liczbę kroków </small>$N$<small> z obiektu `rec.data` oraz końcowy dryf `final_drift`, aby obliczyć wariancję szumu procesu </small>$Q$<small> na pojedynczy krok:</small>


In [ ]:
N_steps = len(rec.data["odometry"])

Q = ...

print(f"Liczba próbek z rec:            N = {N_steps}")
print(f"Zmierzony dryf na mecie:       Δ = {final_drift * 1000:.1f} mm")
print(f"Oszacowany szum procesu:      √Q = {math.sqrt(Q) * 1000:.2f} mm / krok")
print(f"Wariancja szumu procesu:       Q = {Q:.6f} m²  ({Q * 1_000_000:.2f} mm²)")


## Część 2: Czujniki odległości i Filtr Kalmana

**Zadanie 4 – Ruch wahadłowy**

<small>Skupimy się teraz na pozostałych czujnikach, aby zobaczyć, jak radzą sobie w porównaniu do LiDAR-u.</small>

<small>Zaimplementuj płynny ruch wahadłowy robota pod kątem 15° w kierunku ściany i z powrotem, tak aby wykonał dwa pełne okresy oscylacji. Do sterowania prędkością wykorzystaj funkcję okresową:</small>

$$v(t) = A \cdot \sin\left(\frac{2\pi}{T} \cdot t\right), \qquad T = \frac{t_{\text{całk}}}{n}$$

<small>Ruch zapisz w funkcji `pendulum_step(t)`, wywoływanej w każdym kroku pętli – wykorzystamy ją ponownie w kolejnych zadaniach.</small>

<small>Klatka ma wymiary 78 × 78 cm, dlatego ustaw robota ok. 10 cm za środkiem klatki, przodem do ściany, i dobierz amplitudę </small>$A$<small> tak, aby największe przemieszczenie, równe </small>$A \cdot 0{,}39~\text{m/s} \cdot T / \pi$<small>, nie przekroczyło ok. 25 cm.</small>


In [ ]:
SPEED = ...   # A
PERIOD = ...  # T

def pendulum_step(t):
    ...
    robot.set_vector(DriveVector(...))

robot.reset_pose()

clock = Clock(duration=Time.s(10))

rec = SensorRecorder(
    clock,
    title="Ruch wahadłowy: LiDAR vs US vs IR",
    styles={"ultrasonic": "scatter", "ir": "scatter", "lidar": "dashed"},
    lidar=lambda: robot.lidar.beam_front.meters,
    ultrasonic=lambda: robot.us_front.distance.meters,
    ir=lambda: robot.ir_front.distance.meters,
)

with LivePlot(rec), robot:
    for t in clock:
        pendulum_step(t)


<div class="alert alert-block alert-info">
  <small><b>Prześledź wykresy:</b> Co na nich widać? Który z odczytów jest najdokładniejszy, a który najmniej dokładny?</small>
</div>

### Sztuczny szum pomiarowy

<small>Czujniki IR i US naszego robota są dość precyzyjne, a przy ruchu o amplitudzie kilkudziesięciu centymetrów ich szum jest na wykresie praktycznie niewidoczny. Żeby wyraźnie zobaczyć, co z szumem robią kolejne algorytmy, <b>dodamy do odczytów sztuczny szum normalny</b> (gaussowski) o zadanym odchyleniu standardowym </small>$\sigma$<small>:</small>

$$\tilde{d} = d + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, \sigma^2)$$

<small>W ten sam sposób modeluje się czujniki w symulatorach robotów. Czujnikowi US nadajemy większy szum niż IR, aby w fuzji czujniki różniły się wiarygodnością. Od tej chwili odczyty czujników odległości pobieramy funkcją `noisy(sensor, sigma)`, a LiDAR pozostaje niezaszumionym punktem odniesienia.</small>

<small>Komórka poniżej jest gotowa – automatycznie powtarza ruch wahadłowy z Zadania 4 (`pendulum_step`) i rejestruje zaszumione odczyty. Przez pierwsze 2 s robot stoi nieruchomo, więc rozrzut odczytów w tym czasie to wyłącznie szum.</small>

<small>Przed uruchomieniem ustaw robota w tym samym miejscu co w Zadaniu 4 i zmierz <b>miarą zwijaną</b> odległość od przedniej ściany do obudowy czujnika IR oraz do obudowy czujnika US. Wyniki wpisz w komórce pod wykresem.</small>


In [ ]:
rng = np.random.default_rng(seed=18)

SIGMA_IR = 0.02  # odchylenie standardowe sztucznego szumu IR [m]
SIGMA_US = 0.04  # odchylenie standardowe sztucznego szumu US [m]
STILL_TIME = Time.s(2)  # postój na początku przejazdu

def noisy(sensor, sigma):
    """Odczyt czujnika w metrach z dodanym szumem normalnym N(0, sigma²)."""
    return sensor.distance.meters + rng.normal(0.0, sigma)

robot.reset_pose()

clock = Clock(duration=Time.s(12))

rec = SensorRecorder(
    clock,
    title="Postój i ruch wahadłowy z szumem: LiDAR vs US vs IR",
    styles={"ultrasonic": "scatter", "ir": "scatter", "lidar": "dashed"},
    lidar=lambda: robot.lidar.beam_front.meters,
    ultrasonic=lambda: noisy(robot.us_front, SIGMA_US),
    ir=lambda: noisy(robot.ir_front, SIGMA_IR),
)

with LivePlot(rec), robot:
    for t in clock:
        if t < STILL_TIME:
            robot.stop()
        else:
            pendulum_step(Time.s(t.seconds - STILL_TIME.seconds))


In [ ]:
TAPE_IR = ...  # odległość czujnika IR od ściany zmierzona miarą zwijaną [m]
TAPE_US = ...  # odległość czujnika US od ściany zmierzona miarą zwijaną [m]

df_noisy = pd.DataFrame(rec.data)
still = df_noisy[df_noisy["t"] < STILL_TIME.seconds]

print("--- Postój: odczyty czujników ---")
for name, tape, sigma in [("ir", TAPE_IR, SIGMA_IR), ("ultrasonic", TAPE_US, SIGMA_US)]:
    print(
        f"{name:>10}: średnia = {still[name].mean():.3f} m (miara: {tape:.3f} m), "
        f"odch. std. = {still[name].std() * 1000:4.1f} mm (σ = {sigma * 1000:.0f} mm)"
    )
print(f"{'lidar':>10}: średnia = {still['lidar'].mean():.3f} m")


<div class="alert alert-block alert-info">
  <small><b>Porównaj wykresy:</b> Jak zmieniły się odczyty IR i US względem przejazdu z Zadania 4? Który czujnik szumi teraz bardziej? Czy na podstawie pojedynczego odczytu dałoby się wiarygodnie określić położenie robota? Czy średnie odczyty z postoju zgadzają się z pomiarem miarą zwijaną, a odchylenie standardowe z zadanym σ? Skąd bierze się stała różnica między odczytem LiDAR-u a pozostałymi czujnikami?</small>
</div>

**Czym jest szum pomiaru $R$?**

<small>Każdy czujnik jest obarczony losową niedoskonałością aparatury, którą opisujemy statystycznie jako <b>wariancję szumu pomiaru</b> </small>$R$<small>. Wykorzystamy zaszumione pomiary z poprzedniego przejazdu (`rec.data`), aby wyznaczyć szum pomiarowy. Jako punkt odniesienia wykorzystamy LiDAR:</small>

$$e = z_{\text{sensor}} - z_{\text{lidar}}$$

<small>Zamiast zwykłej wariancji (`np.var()`), która jest bezbronna wobec pojedynczych fałszywych ech czujnika ultradźwiękowego (outlierów), stosujemy odporną estymację <b>MAD</b> (odchylenie bezwzględne od mediany):</small>

$$\text{MAD} = \operatorname{median}(|e - \operatorname{median}(e)|), \qquad \sigma \approx 1{,}4826 \cdot \text{MAD}, \qquad R = \sigma^2$$

<small>Wyznaczone w ten sposób wartości </small>$R_{\text{ir}}$<small> oraz </small>$R_{\text{us}}$<small> posłużą jako wagi określające zaufanie do poszczególnych sensorów. Wynik łatwo sprawdzić: </small>$\sqrt{R}$<small> powinno być zbliżone do zadanego </small>$\sigma$<small> (lub nieco większe, bo dochodzi rzeczywisty szum czujnika).</small>

In [ ]:
df = pd.DataFrame(rec.data)
df.head()


In [ ]:
def calc_R(series):
    clean = series.dropna()  # odrzucenie braków pomiaru (np. brak echa, poza zakresem)
    mad = ...  # użyj np.abs, np.median
    return max((1.4826 * mad) ** 2, 1e-6)


In [ ]:
# Wyznacz szum pomiarowy R dla czujników IR i Ultrasonic względem referencyjnego LiDAR-u:
R_ir = ...
R_us = ...

print("\n--- Odporna estymacja szumu losowego R (MAD) ---")
print(f"IR:         √R = {math.sqrt(R_ir) * 1000:5.1f} mm  (sztuczny szum σ = {SIGMA_IR * 1000:.0f} mm)")
print(f"Ultrasonic: √R = {math.sqrt(R_us) * 1000:5.1f} mm  (sztuczny szum σ = {SIGMA_US * 1000:.0f} mm)")


### Filtr Kalmana

<small>Odometria jest płynna, lecz <b>dryfuje</b> (błąd akumuluje się w czasie). Czujniki odległości (ultradźwiękowy, IR) nie dryfują, ale są <b>zaszumione</b>.</small>

<small><b>Filtr Kalmana</b> łączy oba źródła, wyznaczając najbardziej prawdopodobny <b>stan robota</b> </small>$x$<small>, wykonując w każdym kroku </small>$t$<small> naprzemiennie dwa kroki. Punktem wyjścia jest stan z poprzedniego kroku </small>$x_{t-1}$<small> i jego wariancja </small>$P_{t-1}$<small>.</small>

---

#### 1. Krok predykcji (ruch robota z odometrii)
<small>Przesuwamy stan robota o przemieszczenie z kół </small>$\Delta x_t$<small>, otrzymując <b>predykcję</b> </small>$\bar{x}_t$<small>, czyli stan przed uwzględnieniem pomiaru. Niepewność stanu (wariancja </small>$P$<small>) rośnie o szum procesu </small>$Q$<small>:</small>
$$\bar{x}_t = x_{t-1} + \Delta x_t, \qquad \bar{P}_t = P_{t-1} + Q$$

#### 2. Krok korekty (pomiar pozycji $z_t$)
<small>Pomiar </small>$z_t$<small> musi opisywać tę samą wielkość co stan </small>$x$<small>. Ze zmierzonego dystansu do ściany </small>$d_t$<small> wyznaczamy położenie robota jako zbliżenie względem startu (</small>$d_0$<small>):</small>
$$z_t = d_0 - d_t$$

<small>Obliczamy innowację (błąd prognozy) </small>$y_t$<small> oraz <b>wzmocnienie Kalmana</b> </small>$K_t \in [0, 1]$<small>, które określa stopień zaufania do pomiaru:</small>
$$y_t = z_t - \bar{x}_t, \qquad K_t = \frac{\bar{P}_t}{\bar{P}_t + R}$$
<small>Korygujemy stan robota oraz redukujemy jego niepewność. Wynik jest punktem wyjścia dla następnego kroku:</small>
$$x_t = \bar{x}_t + K_t \cdot y_t, \qquad P_t = (1 - K_t) \cdot \bar{P}_t$$

---

<small><b>Intuicja wzmocnienia</b> </small>($K_t$):<br>

* <small>Gdy sensor mocno szumi </small>$(R \gg \bar{P}_t) \implies K_t \to 0$<small>: ignorujemy pomiar, polegamy na odometrii </small>$(x_t \approx \bar{x}_t)$<small>.</small>
* <small>Gdy sensor jest bardzo precyzyjny </small>$(R \ll \bar{P}_t) \implies K_t \to 1$<small>: ufamy pomiarowi </small>$(x_t \approx z_t)$<small>.</small>
* <small>Każda poprawna korekta zmniejsza niepewność stanu: </small>$(1 - K_t) \le 1 \implies P_t < \bar{P}_t$<small>.</small>


**Zadanie 5.1 – Krok predykcji i korekty Filtru Kalmana (1D)**

<small>Uzupełnij implementację kroku predykcji oraz korekty Filtru Kalmana dla pozycji robota </small>$x$<small> na podstawie wzorów powyżej:</small>


In [ ]:
def kalman_predict(x, P, dx, Q):
    """
    Krok predykcji (predict) Filtru Kalmana 1D dla pozycji robota.
    x: aktualna szacowana pozycja robota
    P: wariancja niepewności pozycji
    dx: przemieszczenie robota z odometrii
    Q: wariancja szumu procesu (ruchu)
    """
    x_new = ...
    P_new = ...
    return x_new, P_new


def kalman_update(x, P, z, R):
    """
    Krok korekty (update) Filtru Kalmana 1D dla pozycji robota.
    x: szacowana pozycja robota (przed korektą, z kroku predykcji)
    P: wariancja niepewności pozycji (przed korektą)
    z: pomiar pozycji z czujnika
    R: wariancja szumu pomiarowego
    """
    y = ...
    K = ...
    x_new = ...
    P_new = ...
    return x_new, P_new


**Zadanie 5.2 – Filtr Kalmana w ruchu wahadłowym (1D)**

<small>Wykorzystaj zaimplementowane w Zadaniu 5.1 funkcje `kalman_predict` i `kalman_update` w pętli sterowania. W każdym kroku pętli wykonaj ruch wahadłowy (`pendulum_step`), predykcję z odometrii oraz korektę stanu zaszumionym odczytem czujnika podczerwieni: `noisy(robot.ir_front, SIGMA_IR)`.</small>

In [ ]:
robot.reset_pose()
clock = Clock(duration=Time.s(10))

# --- ZMIENNE FILTRU KALMANA ---
x = 0.0
P = Q
z = 0.0
d0_ir = robot.ir_front.distance.meters
last_odom_x = robot.pose.x.meters

# --- WIZUALIZACJA ---
d0_lidar = robot.lidar.beam_front.meters
rec = SensorRecorder(
    clock,
    title="Filtr Kalmana 1D (IR z szumem) vs Odometria",
    y_label="Pozycja x [m]",
    styles={"ir": "scatter", "kalman": "line", "odometry": "line", "lidar": "dashed"},
    odometry=lambda: robot.pose.x.meters,
    ir=lambda: z,
    kalman=lambda: x,
    lidar=lambda: d0_lidar - robot.lidar.beam_front.meters,
)

with LivePlot(rec), robot:
    for t in clock:
        # Ruch wahadłowy z Zadania 4:
        pendulum_step(t)

        # Predykcja:
        curr_odom_x = robot.pose.x.meters
        dx = ...
        last_odom_x = curr_odom_x
        x, P = kalman_predict(x, P, dx, Q)

        # Pomiar (zaszumiony odczyt IR):
        z = ...

        # Korekta:
        x, P = kalman_update(x, P, z, R_ir)


**Ocena skuteczności filtru – RMSE**

<small>Wykres pokazuje działanie filtru jakościowo. Aby je ocenić liczbowo, porównamy każde źródło z LiDAR-em, obliczając <b>pierwiastek błędu średniokwadratowego</b> (RMSE):</small>

$$\text{RMSE} = \sqrt{\frac{1}{N} \sum_{k=1}^{N} \left(x_k - x_k^{\text{lidar}}\right)^2}$$

<small>Uzupełnij funkcję `rmse` – pomiń próbki, w których brakuje pomiaru (`dropna()`). Wykorzystamy ją również w kolejnych zadaniach.</small>


In [ ]:
def rmse(series, reference):
    e = ...  # błąd względem LiDAR-u, bez brakujących próbek
    return math.sqrt(...)

df = pd.DataFrame(rec.data)
for name in ("odometry", "ir", "kalman"):
    print(f"RMSE {name:>8} vs LiDAR: {rmse(df[name], df['lidar']) * 1000:5.1f} mm")


### Fuzja wielu czujników – w uproszczeniu

<small>Napisany przez nas algorytm Filtru Kalmana pozwala uwzględnić tylko jedno źródło. Z reguły tych źródeł będziemy posiadać wiele – w końcu nasz robot posiada aż sześć czujników oraz skaner laserowy.</small>

<small>Natomiast funkcja w takiej postaci bardzo dobrze nadaje się do nauki, a dołożenie kolejnego czujnika jest tak proste, jak wykonanie sekwencyjnie kolejnych aktualizacji Filtrem Kalmana:</small>

```python
x, P = kalman_predict(x, P, dx, Q)     # Krok predykcji (ruch z odometrii)
x, P = kalman_update(x, P, z_ir, R_ir) # Filtr Kalmana z wejściem IR
x, P = kalman_update(x, P, z_us, R_us) # Filtr Kalmana z wejściem IR oraz US
```

$$\text{Predykcja}(Q) \;\longrightarrow\; \text{Korekta}(z_{\text{IR}}, R_{\text{IR}}) \;\longrightarrow\; \text{Korekta}(z_{\text{US}}, R_{\text{US}})$$


**Zadanie 5.3 – Sekwencyjna fuzja czujników (IR + US)**

<small>Skopiuj kod z Zadania 5.2 i dodaj sekwencyjne wywołanie `kalman_update` dla zaszumionego odczytu US (`noisy(robot.us_front, SIGMA_US)`) nałożone na IR.</small>

<small>Na wspólnym wykresie zestawiamy: odometrię, stan po IR (`kalman_ir`), fuzję obu czujników (`kalman_fusion`) oraz LiDAR jako niezależny punkt odniesienia.</small>

In [ ]:
robot.reset_pose()
clock = Clock(duration=Time.s(10))

# --- ZMIENNE FILTRU KALMANA ---
x = 0.0
P = Q
d0_ir = robot.ir_front.distance.meters
d0_us = robot.us_front.distance.meters
last_odom_x = robot.pose.x.meters

# --- WIZUALIZACJA ---
x_ir = 0.0
x_fusion = 0.0
d0_lidar = robot.lidar.beam_front.meters
rec = SensorRecorder(
    clock,
    title="Sekwencyjna fuzja (IR + US) vs Odometria",
    y_label="Pozycja x [m]",
    styles={"kalman_ir": "line", "kalman_fusion": "line", "odometry": "line", "lidar": "dashed"},
    odometry=lambda: robot.pose.x.meters,
    kalman_ir=lambda: x_ir,
    kalman_fusion=lambda: x_fusion,
    lidar=lambda: d0_lidar - robot.lidar.beam_front.meters,
)

with LivePlot(rec), robot:
    for t in clock:
        ...  # rozpocznij od skopiowania zawartości Zadania 5.2


In [ ]:
df = pd.DataFrame(rec.data)
for name in ("odometry", "kalman_ir", "kalman_fusion"):
    print(f"RMSE {name:>13} vs LiDAR: {rmse(df[name], df['lidar']) * 1000:5.1f} mm")


<div class="alert alert-block alert-info">
  <small><b>Pytanie do przemyślenia:</b> Co gdyby czujnik był zamontowany <b>pod kątem</b>, a nie wprost w kierunku ściany? Czy wówczas również moglibyśmy użyć odczytu bezpośrednio (<i>z</i> &minus; <i>x</i>), czy filtr potrzebuje wiedzieć, w jakiej relacji do osi ruchu znajduje się dany sensor? Odpowiedź na to pytanie znajdziesz poniżej.</small>
</div>


### Pełnoprawna fuzja sensorów (Kalman 2D)

<small>Do tej pory stosowaliśmy uproszczenie w 1D. Pełnoprawny Filtr Kalmana radzi sobie z fuzją wielu źródeł znacznie wydajniej – operuje wprost na rachunku macierzowym, korygując cały stan robota w jednym kroku obliczeniowym.</small>

<small>Bezpośrednim wejściem do algorytmu są macierze, których wymiary wynikają z rozmiaru wektora stanu (</small>$n = 2$<small>) oraz liczby sygnałów pomiarowych (</small>$m = 6$<small>):</small>

* **Stan robota** $\mathbf{x} = \begin{bmatrix} x \\ y \end{bmatrix}$ <small>&ndash; współrzędne pozycji 2D robota na arenie (wymiar </small>$n = 2$<small>).</small>
* **Szum procesu** $\mathbf{Q}_{2 \times 2} = \operatorname{diag}([Q, Q])$ <small>&ndash; niepewność ruchu z odometrii na pojedynczy krok (znamy z Zadania 3).</small>
* **Szum pomiaru** $\mathbf{R}_{6 \times 6} = \operatorname{diag}([R, \dots, R])$ <small>&ndash; szumy pomiarowe 6 sensorów (znamy z Zadania 4).</small>
* **Macierz obserwacji** $\mathbf{H}_{6 \times 2}$ <small>&ndash; określa, <b>w jaki sposób każdy sensor wpływa na stan</b> (</small>$\mathbf{z} = \mathbf{H}\mathbf{x} + \mathbf{v}$<small>). Każdy wiersz reprezentuje jedno źródło pomiaru, a kolumny przypisują sensor do mierzonej współrzędnej (</small>$x$<small> lub </small>$y$<small>):</small>

$$\mathbf{H} = \left. \begin{bmatrix} 
1 & 0 \\
\vdots & \vdots 
\end{bmatrix} \right\} \text{6 wierszy (czujników)}
\quad
\begin{matrix} 
\leftarrow \text{czujnik przedni (wzrost } x\text{)} \\
\dots 
\end{matrix}$$

<small>Skaner LiDAR nie bierze udziału w filtracji – posłuży na wykresie jako niezależny punkt odniesienia.</small>


**Zadanie 6 – Fuzja odometrii i 6 czujników pokładowych**

<small>Dokończ definicję wektora źródeł pomiarowych </small>`z`<small> (odczyty znormalizowane o odległość bazową </small>$d_0$<small>) oraz uzupełnij macierze </small>`Q_mat`<small>, </small>`R_mat`<small> i </small>`H`<small>. Tak jak w Zadaniu 5, korzystaj z zaszumionych odczytów </small>`noisy(...)`<small>:</small>


In [ ]:
d0_ir_f, d0_ir_l, d0_ir_r = None, None, None
d0_us_f, d0_us_l, d0_us_r = None, None, None

def sensors_reset():
    global d0_ir_f, d0_ir_l, d0_ir_r, d0_us_f, d0_us_l, d0_us_r
    d0_ir_f, d0_ir_l, d0_ir_r = robot.ir_front.distance.meters, robot.ir_left.distance.meters, robot.ir_right.distance.meters
    d0_us_f, d0_us_l, d0_us_r = robot.us_front.distance.meters, robot.us_left.distance.meters, robot.us_right.distance.meters

robot.reset_pose()
sensors_reset()

z = [
    lambda: ...,  # IR przód
    lambda: ...,  # IR lewy
    lambda: ...,  # IR prawy
    lambda: ...,  # US przód
    lambda: ...,  # US lewy
    lambda: ...,  # US prawy
]

Q_mat = np.diag([..., ...])

R_mat = np.diag([..., ..., ..., ..., ..., ...])

H = np.array([
    ...,  # IR przód
    ...,  # IR lewy
    ...,  # IR prawy
    ...,  # US przód
    ...,  # US lewy
    ...,  # US prawy
])


<small>Uruchamiamy jeden przejazd testowy na trasie kwadratu. Rejestrujemy jednocześnie na jednym wykresie dryfującą odometrię, stan robota z Filtru Kalmana 2D (fuzja 6 czujników) oraz niezależny punkt odniesienia z LiDAR-u:</small>


In [ ]:
robot.reset_pose()
lidar_reset()
sensors_reset()
kalman = Kalman2d(robot, z=z, Q=Q_mat, R=R_mat, H=H)
rec = PoseRecorder(
    odometry=lambda: (robot.pose.x.meters, robot.pose.y.meters),
    kalman=lambda: kalman.state,
    lidar=lambda: (lidar_get_x().meters, lidar_get_y().meters),
    title="Odometria, Filtr Kalmana, LiDAR"
)
with LivePlot(rec), robot:
    drive_square()


In [ ]:
def rmse_2d(path, reference):
    """RMSE odległości między dwiema trajektoriami 2D (listy punktów (x, y))."""
    a, b = np.asarray(path, dtype=float), np.asarray(reference, dtype=float)
    n = min(len(a), len(b))
    return math.sqrt(np.nanmean(np.sum((a[:n] - b[:n]) ** 2, axis=1)))

for name in ("odometry", "kalman"):
    print(f"RMSE {name:>8} vs LiDAR: {rmse_2d(rec.data[name], rec.data['lidar']) * 1000:5.1f} mm")


### <small>Co się dzieje przy złym strojeniu parametrów?</small>

<small>Wzmocnienie Kalmana </small>$K$<small> zależy od stosunku niepewności ruchu </small>$Q$<small> do szumu pomiaru </small>$R$<small>:</small>
* <small><b>Za duże </b></small>$R$<small><b> (lub za małe </b></small>$Q$<small><b>):</b> filtr nie ufa pomiarom, polega na odometrii i przejmuje jej narastający dryf.</small>
* <small><b>Za małe </b></small>$R$<small><b> (lub za duże </b></small>$Q$<small><b>):</b> filtr ufa bezkrytycznie każdemu odczytowi &rarr; stan robota drży od szumu aparatury.</small>

<small>Uruchamiamy trzy filtry 2D jednocześnie podczas jednego przejazdu po kwadracie. Jako punkt odniesienia rysujemy też położenie z LiDAR-u, a pod wykresem wypisujemy RMSE każdego filtru:</small>
* <small>`kf_optimal` &ndash; dobrze dobrane parametry (stabilny, domknięty kwadrat),</small>
* <small>`kf_trust_odom` &ndash; </small>$R \times 1000$<small> (filtr ignoruje czujniki i dryfuje z kołami),</small>
* <small>`kf_trust_sensor` &ndash; </small>$R \times 0{,}001$<small> (filtr jest nerwowy i szarpie pozycją przy każdym szumie czujników).</small>


In [ ]:
robot.reset_pose()
lidar_reset()
sensors_reset()

kf_optimal      = Kalman2d(robot, z=z, Q=Q_mat, R=R_mat, H=H)
kf_trust_odom   = Kalman2d(robot, z=z, Q=Q_mat, R=R_mat * 1000, H=H)
kf_trust_sensor = Kalman2d(robot, z=z, Q=Q_mat * 1000, R=R_mat * 0.001, H=H)

rec_tuning_2d = PoseRecorder(
    optimal=lambda: kf_optimal.state,
    trust_odom=lambda: kf_trust_odom.state,
    trust_sensor=lambda: kf_trust_sensor.state,
    lidar=lambda: (lidar_get_x().meters, lidar_get_y().meters),
    title="Wpływ strojenia Q i R w 2D"
)

with LivePlot(rec_tuning_2d), robot:
    drive_square()


In [ ]:
for name in ("optimal", "trust_odom", "trust_sensor"):
    print(f"RMSE {name:>12} vs LiDAR: {rmse_2d(rec_tuning_2d.data[name], rec_tuning_2d.data['lidar']) * 1000:5.1f} mm")
